# LeetCode 1448 — Count Good Nodes in Binary Tree bằng Python

Link đề: https://leetcode.com/problems/count-good-nodes-in-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def goodNodes(self, root: TreeNode) -> int:
        ...
```

## 1. Mục tiêu học

1. Tự dựng `TreeNode` và hàm **dựng cây từ mảng level-order** (có `None`) để test bài cây trên máy.
2. Hiểu định nghĩa **nút tốt**: không có nút nào trên đường từ gốc tới nó **lớn hơn** nó ⇔ `node.val ≥ max(đường đi từ gốc tới cha)`.
3. Kỹ thuật **truyền thông tin từ trên xuống** (top-down): DFS mang theo `max` của đường đi.
4. Viết cùng một ý tưởng bằng **DFS đệ quy**, **DFS lặp bằng stack** và **BFS bằng queue**.
5. Tránh lỗi: chỉ so với cha, dùng `>` thay `≥`, khởi tạo max bằng 0 khi có số âm, dùng biến max toàn cục không "hoàn tác" khi quay lui.

## 2. Đề bài & ví dụ

Cho gốc `root` của một cây nhị phân. Một nút `X` được gọi là **tốt (good)** nếu trên đường đi từ gốc tới `X` **không có nút nào có giá trị lớn hơn** `X`. Trả về số nút tốt.

**Ví dụ 1:** `root = [3,1,4,3,null,1,5]` → `4`. Các nút tốt: gốc `3`, nút `4` (đường 3→4), nút `5` (3→4→5), nút `3` ở dưới (3→1→3).

```
        3
      /   \
     1     4
    /     / \
   3     1   5
```

**Ví dụ 2:** `root = [3,3,null,4,2]` → `3`. Nút `2` (đường 3→3→2) không tốt vì có `3 > 2`.

**Ví dụ 3:** `root = [1]` → `1` (gốc luôn tốt).

**Ràng buộc:** số nút trong `[1, 10^5]`; giá trị mỗi nút trong `[-10^4, 10^4]`.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List, Optional

examples = [
    ([3, 1, 4, 3, None, 1, 5], 4),
    ([3, 3, None, 4, 2], 3),
    ([1], 1),
]
for arr, want in examples:
    print(f"root={arr} -> kỳ vọng {want}")

root=[3, 1, 4, 3, None, 1, 5] -> kỳ vọng 4
root=[3, 3, None, 4, 2] -> kỳ vọng 3
root=[1] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Lớp `TreeNode` và dựng cây từ level-order

LeetCode biểu diễn cây bằng mảng theo **từng tầng** (level-order), `None` là chỗ trống. Hàm dựng dùng một hàng đợi: lấy nút cha ra, gán lần lượt con trái/phải từ mảng.

In [2]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(arr):
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0])
    q = deque([root])
    i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root

def to_list(root):
    # chuyển ngược cây -> mảng level-order (bỏ None thừa ở cuối) để kiểm tra build_tree
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node:
            out.append(node.val); q.append(node.left); q.append(node.right)
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

r = build_tree([3, 1, 4, 3, None, 1, 5])
print(r.val, r.left.val, r.right.val, r.left.left.val, r.right.left.val, r.right.right.val)
print(to_list(r))

3 1 4 3 1 5
[3, 1, 4, 3, None, 1, 5]


### 3.2 Vẽ cây dạng chữ để dễ hình dung

Một hàm in cây nằm ngang (gốc bên trái, con phải ở trên) giúp đối chiếu trace.

In [3]:
def show(root, indent=""):
    if root is None:
        return
    show(root.right, indent + "    ")
    print(indent + str(root.val))
    show(root.left, indent + "    ")

show(build_tree([3, 1, 4, 3, None, 1, 5]))

        5
    4
        1
3
    1
        3


### 3.3 Duyệt cây: đệ quy, stack, queue

- **DFS đệ quy:** hàm gọi chính nó cho con trái và con phải.
- **DFS lặp:** dùng `list` làm stack, đẩy `(nút, thông tin)`.
- **BFS:** dùng `deque` làm queue, duyệt theo tầng.

Cả ba đều thăm mỗi nút đúng một lần.

In [4]:
root = build_tree([3, 1, 4, 3, None, 1, 5])

def preorder(node):
    return [] if node is None else [node.val] + preorder(node.left) + preorder(node.right)

stack, dfs_order = [root], []
while stack:
    node = stack.pop()
    dfs_order.append(node.val)
    for child in (node.right, node.left):    # đẩy phải trước để trái ra trước
        if child:
            stack.append(child)

q, bfs_order = deque([root]), []
while q:
    node = q.popleft()
    bfs_order.append(node.val)
    q.extend(c for c in (node.left, node.right) if c)
print("đệ quy:", preorder(root), "| stack:", dfs_order, "| BFS:", bfs_order)

đệ quy: [3, 1, 3, 4, 1, 5] | stack: [3, 1, 3, 4, 1, 5] | BFS: [3, 1, 4, 3, 1, 5]


### 3.4 `float('-inf')` — giá trị nhỏ hơn mọi số

Khi cần "max của tập rỗng" làm giá trị khởi đầu, dùng `float('-inf')` (hoặc một số nhỏ hơn ràng buộc, như `-10**4 - 1`). **Không** dùng `0` khi dữ liệu có số âm.

In [5]:
NEG = float('-inf')
print(NEG < -10**9, max(NEG, -5), -3 >= NEG)
print("max với khởi tạo 0 cho dãy toàn âm:", max([0, -5, -2]), "<- sai nếu dữ liệu toàn âm")

True -5 True
max với khởi tạo 0 cho dãy toàn âm: 0 <- sai nếu dữ liệu toàn âm


## 4. Ý tưởng nền tảng / trực giác

"Không có nút nào trên đường đi lớn hơn `X`" ⇔ `X.val ≥ max(các nút từ gốc đến cha của X)`.

Tính tay ví dụ 1, đi từ gốc xuống, ghi `max` của đường đi **trước** khi tới nút:

| nút | đường đi tới cha | max trước | val ≥ max? | tốt? |
|---|---|---|---|---|
| 3 (gốc) | (rỗng) | −∞ | ✔ | ✔ |
| 1 | 3 | 3 | 1 ≥ 3 ✘ | ✘ |
| 3 (lá trái) | 3,1 | 3 | 3 ≥ 3 ✔ | ✔ |
| 4 | 3 | 3 | ✔ | ✔ |
| 1 (dưới 4) | 3,4 | 4 | ✘ | ✘ |
| 5 | 3,4 | 4 | ✔ | ✔ |

Tổng 4 ✔. Quan sát: thông tin cần truyền xuống chỉ là **một con số** — `max` của đường đi. Khi đi xuống con, `max mới = max(max cũ, node.val)`.

In [6]:
def walk(node, mx, path):
    if node is None:
        return
    good = node.val >= mx
    print(f"nút {node.val:2d} | đường {path} | max trước = {mx} | tốt? {good}")
    walk(node.left, max(mx, node.val), path + [node.val])
    walk(node.right, max(mx, node.val), path + [node.val])

walk(build_tree([3, 1, 4, 3, None, 1, 5]), float('-inf'), [])

nút  3 | đường [] | max trước = -inf | tốt? True
nút  1 | đường [3] | max trước = 3 | tốt? False
nút  3 | đường [3, 1] | max trước = 3 | tốt? True
nút  4 | đường [3] | max trước = 3 | tốt? True
nút  1 | đường [3, 4] | max trước = 4 | tốt? False
nút  5 | đường [3, 4] | max trước = 4 | tốt? True


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** lưu **cả đường đi** (list giá trị) từ gốc tới mỗi nút, rồi kiểm tra trực tiếp định nghĩa: `all(v <= node.val for v in path)`.

Mỗi lần đi xuống ta tạo bản sao `path + [node.val]` và kiểm tra lại toàn bộ đường ⇒ tốn `O(h)` cho mỗi nút.

In [7]:
def good_brute(root: Optional[TreeNode]) -> int:
    count = 0
    stack = [(root, [])] if root else []
    while stack:
        node, path = stack.pop()
        if all(v <= node.val for v in path):     # định nghĩa trực tiếp
            count += 1
        for child in (node.left, node.right):
            if child:
                stack.append((child, path + [node.val]))
    return count

for arr, want in examples:
    print(good_brute(build_tree(arr)), "kỳ vọng", want)

4 kỳ vọng 4
3 kỳ vọng 3
1 kỳ vọng 1


**Độ phức tạp:** Time `O(n·h)` (h = chiều cao, cây lệch thì `h = n` ⇒ `O(n²)`), Space `O(n·h)` vì mỗi phần tử trong stack giữ một bản sao đường đi.

**Nút thắt:** ta chỉ cần `max(path)`, không cần cả `path`. Lưu `max` thay cho list ⇒ `O(1)` mỗi nút.

In [8]:
def chain(n):
    # cây lệch phải 1 -> 2 -> ... -> n (level-order: [1, None, 2, None, 3, ...])
    arr = [1]
    for v in range(2, n + 1):
        arr += [None, v]
    return build_tree(arr)

t = time.perf_counter()
print(good_brute(chain(3000)), f"(brute, cây lệch 3000 nút: {time.perf_counter()-t:.2f}s)")

3000 (brute, cây lệch 3000 nút: 0.12s)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** thay đường đi bằng **một con số** `mx = max(đường đi từ gốc tới cha)`.

- **Nếu** `node.val ≥ mx` **thì** `node` tốt.
- Khi đi xuống con: `mx_con = max(mx, node.val)`.
- Gốc: `mx = −∞` (gốc luôn tốt).

**Invariant:** mỗi khi xử lý một nút, `mx` đúng bằng giá trị lớn nhất trên đường từ gốc tới cha của nó. Nhờ truyền `mx` theo **tham số** (hoặc kèm trong stack/queue), mỗi nhánh có bản `mx` riêng — không bị nhánh anh em làm ảnh hưởng.

Thông tin chảy **từ trên xuống** (top-down), trái ngược với các bài "chiều cao", "tổng cây con" chảy từ dưới lên.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS đệ quy mang theo `mx`

**Giải thích từng đoạn:**
1. `dfs(node, mx)` trả về số nút tốt trong cây con `node`.
2. Nút rỗng ⇒ 0.
3. `good = 1 if node.val >= mx else 0`; `mx2 = max(mx, node.val)`.
4. Trả về `good + dfs(left, mx2) + dfs(right, mx2)`.

In [9]:
def good_dfs(root: Optional[TreeNode]) -> int:
    def dfs(node, mx):
        if node is None:
            return 0
        good = 1 if node.val >= mx else 0
        mx2 = max(mx, node.val)
        return good + dfs(node.left, mx2) + dfs(node.right, mx2)
    return dfs(root, float('-inf'))

for arr, want in examples:
    print(good_dfs(build_tree(arr)), "kỳ vọng", want)

4 kỳ vọng 4
3 kỳ vọng 3
1 kỳ vọng 1


**Code nộp LeetCode (cách 2)** — nâng giới hạn đệ quy vì cây có thể lệch tới `10^5` tầng:

In [10]:
class SolutionRecursive:
    def goodNodes(self, root: TreeNode) -> int:
        sys.setrecursionlimit(max(sys.getrecursionlimit(), 2 * 10**5 + 10))

        def dfs(node, mx):
            if node is None:
                return 0
            mx2 = max(mx, node.val)
            return (node.val >= mx) + dfs(node.left, mx2) + dfs(node.right, mx2)

        return dfs(root, root.val)          # gốc: so với chính nó -> luôn tốt

print(SolutionRecursive().goodNodes(build_tree([3, 1, 4, 3, None, 1, 5])))

4


`(node.val >= mx)` là `True/False`, cộng với số nguyên thì thành `1/0`. Khởi tạo `mx = root.val` cũng đúng vì gốc so với chính nó.

**Độ phức tạp:** Time `O(n)`, Space `O(h)` stack đệ quy.

### 7.2 Cách 3 — DFS lặp bằng stack `(nút, mx)`

**Giải thích:** thay đệ quy bằng stack tự quản lý; mỗi phần tử mang **bản `mx` riêng** của nó nên không cần "hoàn tác". Không lo giới hạn đệ quy.

In [11]:
def good_stack(root: Optional[TreeNode]) -> int:
    if root is None:
        return 0
    count = 0
    stack = [(root, root.val)]
    while stack:
        node, mx = stack.pop()
        if node.val >= mx:
            count += 1
        mx2 = max(mx, node.val)
        if node.left:
            stack.append((node.left, mx2))
        if node.right:
            stack.append((node.right, mx2))
    return count

for arr, want in examples:
    print(good_stack(build_tree(arr)), "kỳ vọng", want)

4 kỳ vọng 4
3 kỳ vọng 3
1 kỳ vọng 1


**Code nộp LeetCode (cách 3)** — xem mục 9 (đây là lời giải chính). **Độ phức tạp:** Time `O(n)`, Space `O(h)` đến `O(n)` cho stack.

### 7.3 Cách 4 — BFS theo tầng với queue `(nút, mx)`

**Giải thích:** giống cách 3 nhưng dùng `deque` và `popleft` — duyệt theo tầng. Kết quả không đổi vì điều kiện "tốt" chỉ phụ thuộc đường đi từ gốc, không phụ thuộc thứ tự duyệt.

In [12]:
def good_bfs(root: Optional[TreeNode]) -> int:
    if root is None:
        return 0
    count = 0
    q = deque([(root, root.val)])
    while q:
        node, mx = q.popleft()
        if node.val >= mx:
            count += 1
        mx2 = max(mx, node.val)
        for child in (node.left, node.right):
            if child:
                q.append((child, mx2))
    return count

class SolutionBFS:
    def goodNodes(self, root: TreeNode) -> int:
        count, q = 0, deque([(root, root.val)])
        while q:
            node, mx = q.popleft()
            if node.val >= mx:
                count += 1
                mx = node.val
            for child in (node.left, node.right):
                if child:
                    q.append((child, mx))
        return count

for arr, want in examples:
    print(good_bfs(build_tree(arr)), SolutionBFS().goodNodes(build_tree(arr)), "kỳ vọng", want)

4 4 kỳ vọng 4
3 3 kỳ vọng 3
1 1 kỳ vọng 1


Trong `SolutionBFS` ta gộp: nếu nút tốt thì `node.val ≥ mx` nên `max(mx, node.val) = node.val` ⇒ gán `mx = node.val`; nếu không tốt thì `mx` giữ nguyên.

**Độ phức tạp:** Time `O(n)`, Space `O(w)` với `w` là độ rộng lớn nhất của một tầng (tối đa ~`n/2`).

So sánh tốc độ trên cây `10^5` nút (cây ngẫu nhiên và cây lệch):

In [13]:
def random_tree(n, rng, lo=-10**4, hi=10**4):
    # cây nhị phân ngẫu nhiên n nút: gắn từng nút mới vào một chỗ trống ngẫu nhiên
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, 0), (root, 1)]
    for _ in range(n - 1):
        k = rng.randrange(len(slots))
        slots[k], slots[-1] = slots[-1], slots[k]
        parent, side = slots.pop()
        node = TreeNode(rng.randint(lo, hi))
        if side == 0:
            parent.left = node
        else:
            parent.right = node
        slots += [(node, 0), (node, 1)]
    return root

rng = random.Random(1448)
big_random = random_tree(10**5, rng)
big_chain = chain(10**5)
for tname, tree in [("ngẫu nhiên", big_random), ("lệch phải", big_chain)]:
    for name, f in [("stack", good_stack), ("bfs", good_bfs)]:
        t = time.perf_counter()
        res = f(tree)
        print(f"{tname:10} {name:6} -> {res:6d} ({time.perf_counter()-t:.3f}s)")

ngẫu nhiên stack  ->   3305 (0.039s)
ngẫu nhiên bfs    ->   3305 (0.048s)
lệch phải  stack  -> 100000 (0.013s)
lệch phải  bfs    -> 100000 (0.017s)


## 8. TRACE từng bước

Trace cách 3 (stack) trên ví dụ 1. Stack in ra dạng `(giá trị nút, mx)`; con phải được đẩy sau nên ra trước.

| bước | lấy ra (val, mx) | tốt? | count | đẩy vào |
|---|---|---|---|---|
| 1 | (3, 3) | ✔ | 1 | (1,3), (4,3) |
| 2 | (4, 3) | ✔ | 2 | (1,4), (5,4) |
| 3 | (5, 4) | ✔ | 3 | – |
| 4 | (1, 4) | ✘ | 3 | – |
| 5 | (1, 3) | ✘ | 3 | (3,3) |
| 6 | (3, 3) | ✔ | 4 | – |

In [14]:
def trace_stack(root):
    count, step = 0, 0
    stack = [(root, root.val)]
    while stack:
        step += 1
        node, mx = stack.pop()
        good = node.val >= mx
        count += good
        mx2 = max(mx, node.val)
        pushed = []
        for child in (node.left, node.right):
            if child:
                stack.append((child, mx2))
                pushed.append((child.val, mx2))
        print(f"bước {step}: lấy ({node.val}, mx={mx}) -> {'tốt' if good else 'không'} | count={count}"
              f" | đẩy {pushed} | stack={[(n.val, m) for n, m in stack]}")
    return count

trace_stack(build_tree([3, 1, 4, 3, None, 1, 5]))

bước 1: lấy (3, mx=3) -> tốt | count=1 | đẩy [(1, 3), (4, 3)] | stack=[(1, 3), (4, 3)]
bước 2: lấy (4, mx=3) -> tốt | count=2 | đẩy [(1, 4), (5, 4)] | stack=[(1, 3), (1, 4), (5, 4)]
bước 3: lấy (5, mx=4) -> tốt | count=3 | đẩy [] | stack=[(1, 3), (1, 4)]
bước 4: lấy (1, mx=4) -> không | count=3 | đẩy [] | stack=[(1, 3)]
bước 5: lấy (1, mx=3) -> không | count=3 | đẩy [(3, 3)] | stack=[(3, 3)]
bước 6: lấy (3, mx=3) -> tốt | count=4 | đẩy [] | stack=[]


4

Trace cách 2 (đệ quy) trên ví dụ 2 — thụt lề theo độ sâu, in giá trị trả về của từng cây con:

In [15]:
def trace_dfs(root):
    def dfs(node, mx, d):
        if node is None:
            return 0
        good = node.val >= mx
        mx2 = max(mx, node.val)
        print("  " * d + f"vào {node.val} (mx={mx}) -> {'tốt' if good else 'không'}")
        res = good + dfs(node.left, mx2, d + 1) + dfs(node.right, mx2, d + 1)
        print("  " * d + f"cây con {node.val} có {res} nút tốt")
        return res
    return dfs(root, root.val, 0)

trace_dfs(build_tree([3, 3, None, 4, 2]))

vào 3 (mx=3) -> tốt
  vào 3 (mx=3) -> tốt
    vào 4 (mx=3) -> tốt
    cây con 4 có 1 nút tốt
    vào 2 (mx=3) -> không
    cây con 2 có 0 nút tốt
  cây con 3 có 2 nút tốt
cây con 3 có 3 nút tốt


3

## 9. Code nộp LeetCode

Lời giải chính (cách 3 — DFS lặp, không lo giới hạn đệ quy), đúng chữ ký `goodNodes(self, root)`. Trên LeetCode lớp `TreeNode` đã có sẵn; ở đây ta dùng lớp tự dựng phía trên.

In [16]:
class Solution:
    def goodNodes(self, root: TreeNode) -> int:
        count = 0
        stack = [(root, root.val)]
        while stack:
            node, mx = stack.pop()
            if node.val >= mx:
                count += 1
                mx = node.val
            if node.left:
                stack.append((node.left, mx))
            if node.right:
                stack.append((node.right, mx))
        return count

for arr, want in examples:
    assert Solution().goodNodes(build_tree(arr)) == want
print("Solution: 3 ví dụ đề OK")

Solution: 3 ví dụ đề OK


## 10. Vì sao thuật toán đúng

**Bổ đề:** khi một phần tử `(node, mx)` được lấy ra khỏi stack/queue (hoặc khi gọi `dfs(node, mx)`), `mx` bằng max các giá trị trên đường từ gốc tới **cha** của `node` (với gốc: `mx = root.val`, tương đương "không có ai lớn hơn").

*Chứng minh quy nạp theo độ sâu:* đúng với gốc. Nếu đúng với `node` (giá trị `mx`), thì con của nó nhận `max(mx, node.val)` = max đường từ gốc tới `node` = max đường từ gốc tới **cha của con** ✔.

**Kết luận:** `node` tốt ⇔ không có giá trị nào trên đường tới cha lớn hơn `node.val` ⇔ `max(...) ≤ node.val` ⇔ `node.val ≥ mx`. Mỗi nút được xét đúng một lần nên `count` đúng bằng số nút tốt.

Vì mỗi phần tử mang **bản `mx` riêng**, các nhánh anh em không ảnh hưởng nhau (đây là chỗ bản "biến toàn cục" ở lỗi 4 sai).

## 11. Phân tích Time / Space

Gọi `n` là số nút, `h` là chiều cao, `w` là độ rộng lớn nhất.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute (lưu đường đi) | `O(n·h)` ⊆ `O(n²)` | `O(n·h)` | sao chép đường đi |
| 2. DFS đệ quy | `O(n)` | `O(h)` stack | cần nâng recursion limit |
| 3. DFS stack | `O(n)` | `O(h)`…`O(n)` | an toàn trong Python |
| 4. BFS queue | `O(n)` | `O(w)` | duyệt theo tầng |

Cây cân bằng: `h ≈ log n`; cây lệch: `h = n`.

## 12. Bộ test

Ví dụ đề, edge case (1 nút, giá trị âm, toàn bằng nhau, tăng dần/giảm dần theo chiều sâu, cây lệch trái), đối chiếu ngẫu nhiên với brute force (seed cố định, giá trị trong ràng buộc).

In [17]:
ALL = {
    "brute": good_brute,
    "dfs": good_dfs,
    "stack": good_stack,
    "bfs": good_bfs,
    "SolutionRecursive": lambda r: SolutionRecursive().goodNodes(r),
    "SolutionBFS": lambda r: SolutionBFS().goodNodes(r),
    "Solution": lambda r: Solution().goodNodes(r),
}
edge = [
    ([-5], 1),
    ([-1, -2, -1], 2),
    ([2, 2, 2, 2, 2, 2, 2], 7),
    ([1, None, 2, None, 3, None, 4], 4),     # tăng dần theo chiều sâu: mọi nút tốt
    ([4, 3, None, 2, None, 1], 1),           # giảm dần: chỉ gốc
    ([3, 1, None, 2], 1),                    # 2 > cha 1 nhưng < gốc 3
    ([1, 5, 2], 3),
    ([-10000, 10000, -10000], 3),
]
for arr, want in examples + edge:
    for name, f in ALL.items():
        got = f(build_tree(arr))
        assert got == want, (name, arr, got, want)
print("Ví dụ đề + edge case: OK cho", len(ALL), "cách")

Ví dụ đề + edge case: OK cho 7 cách


In [18]:
rng = random.Random(2024)
for _ in range(800):
    n = rng.randint(1, 40)
    span = rng.choice([2, 5, 10**4])
    tree = random_tree(n, rng, -span, span)
    want = good_brute(tree)
    for name, f in ALL.items():
        assert f(tree) == want, (name, to_list(tree))
print("800 cây ngẫu nhiên: mọi cách khớp brute force")
assert good_stack(big_random) == good_bfs(big_random) == Solution().goodNodes(big_random)
assert Solution().goodNodes(big_chain) == 10**5
print("Cây 10^5 nút: các cách khớp; cây lệch tăng dần có 10^5 nút tốt")

800 cây ngẫu nhiên: mọi cách khớp brute force


Cây 10^5 nút: các cách khớp; cây lệch tăng dần có 10^5 nút tốt


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ so với **cha**, không so với cả đường đi

Đề nói "không có nút nào trên đường đi lớn hơn", không chỉ cha.

In [19]:
def wrong_parent_only(root):
    count = 0
    stack = [(root, root.val)]          # (nút, giá trị cha)
    while stack:
        node, parent_val = stack.pop()
        if node.val >= parent_val:
            count += 1
        for c in (node.left, node.right):
            if c:
                stack.append((c, node.val))   # SAI: truyền giá trị cha, không phải max
    return count

case = [3, 1, None, 2]
print("Sai :", wrong_parent_only(build_tree(case)), "| Đúng:", Solution().goodNodes(build_tree(case)))
print("Nút 2 lớn hơn cha 1 nhưng nhỏ hơn gốc 3 -> không tốt.")

Sai : 2 | Đúng: 1
Nút 2 lớn hơn cha 1 nhưng nhỏ hơn gốc 3 -> không tốt.


**Sửa:** truyền `max(mx, node.val)` xuống con.

### Lỗi 2 — Dùng `>` thay vì `>=`

Giá trị **bằng** max vẫn tốt (không có nút nào **lớn hơn**).

In [20]:
def wrong_strict(root):
    count = 0
    stack = [(root, float('-inf'))]
    while stack:
        node, mx = stack.pop()
        if node.val > mx:                    # SAI
            count += 1
        for c in (node.left, node.right):
            if c:
                stack.append((c, max(mx, node.val)))
    return count

for arr in ([3, 3, None, 4, 2], [3, 1, 4, 3, None, 1, 5]):
    print(f"{arr}: Sai {wrong_strict(build_tree(arr))} | Đúng {Solution().goodNodes(build_tree(arr))}")

[3, 3, None, 4, 2]: Sai 2 | Đúng 3
[3, 1, 4, 3, None, 1, 5]: Sai 3 | Đúng 4


**Sửa:** `node.val >= mx`.

### Lỗi 3 — Khởi tạo `mx = 0` khi cây có giá trị âm

Giá trị có thể âm tới `−10^4`. Với `mx = 0`, gốc âm bị coi là không tốt.

In [21]:
def wrong_init_zero(root):
    def dfs(node, mx):
        if node is None:
            return 0
        return (node.val >= mx) + dfs(node.left, max(mx, node.val)) + dfs(node.right, max(mx, node.val))
    return dfs(root, 0)                      # SAI

for arr in ([-1, -2, -1], [-5]):
    print(f"{arr}: Sai {wrong_init_zero(build_tree(arr))} | Đúng {Solution().goodNodes(build_tree(arr))}")

[-1, -2, -1]: Sai 0 | Đúng 2
[-5]: Sai 0 | Đúng 1


**Sửa:** khởi tạo `mx = float('-inf')` hoặc `mx = root.val`.

### Lỗi 4 — Dùng một biến `max` dùng chung, không hoàn tác khi quay lui

Cập nhật `self.mx` khi đi xuống nhánh trái rồi quên khôi phục ⇒ nhánh phải bị so với max của nhánh trái (vốn không nằm trên đường đi của nó).

In [22]:
class WrongShared:
    def goodNodes(self, root):
        self.mx = float('-inf')
        self.count = 0
        def dfs(node):
            if node is None:
                return
            if node.val >= self.mx:
                self.count += 1
                self.mx = node.val           # SAI: không khôi phục khi quay lên
            dfs(node.left)
            dfs(node.right)
        dfs(root)
        return self.count

case = [1, 5, 2]
print("Sai :", WrongShared().goodNodes(build_tree(case)), "| Đúng:", Solution().goodNodes(build_tree(case)))
print("Nút 2 bị so với 5 ở nhánh trái, dù đường đi của nó chỉ là 1 -> 2.")

Sai : 2 | Đúng: 3
Nút 2 bị so với 5 ở nhánh trái, dù đường đi của nó chỉ là 1 -> 2.


**Sửa:** truyền `mx` qua tham số (mỗi lời gọi có bản riêng), hoặc lưu `old = self.mx` trước khi gán và khôi phục `self.mx = old` sau khi duyệt xong hai con.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Điều kiện phụ thuộc **đường đi từ gốc tới nút** ⇒ **truyền thông tin từ trên xuống** qua tham số DFS (max, min, tổng, độ sâu, bitmask…).
- Chỉ cần **tóm tắt** của đường đi (ở đây là max) ⇒ không lưu cả list.
- Python + cây có thể sâu `10^5` ⇒ ưu tiên **stack lặp** hoặc nâng `sys.setrecursionlimit`.
- Đếm bằng `True/False` cộng trực tiếp: `count += node.val >= mx`.
- Test bài cây: luôn có `build_tree` từ level-order và vài cây lệch, cây toàn bằng nhau, giá trị âm.

## 15. Biến thể của bài

1. **Nút "kỷ lục" nghiêm ngặt:** lớn hơn **hẳn** mọi tổ tiên (dùng `>`; gốc vẫn tính) — bài tập 4.
2. **Maximum Difference Between Node and Ancestor** (LeetCode 1026): mang cả `min` và `max` của đường đi — bài tập 5.
3. **Path Sum** (LeetCode 112/113): mang **tổng** đường đi.
4. **Pseudo-Palindromic Paths** (LeetCode 1457): mang **bitmask chẵn lẻ** của đường đi.
5. **Sum Root to Leaf Numbers** (LeetCode 129): mang số đang tạo `cur·10 + val`.

In [23]:
def sum_root_to_leaf(root):
    # biến thể 5: minh hoạ cùng khuôn "mang thông tin xuống" bằng stack
    total, stack = 0, [(root, 0)]
    while stack:
        node, cur = stack.pop()
        cur = cur * 10 + node.val
        if not node.left and not node.right:
            total += cur
        for c in (node.left, node.right):
            if c:
                stack.append((c, cur))
    return total

print(sum_root_to_leaf(build_tree([1, 2, 3])), sum_root_to_leaf(build_tree([4, 9, 0, 5, 1])))

25 1026


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Lưu cả đường đi, kiểm tra `all(...)` | `O(n·h)` | `O(n·h)` | Kiểm chứng |
| 2. DFS đệ quy | Tham số `mx` | `O(n)` | `O(h)` | Code ngắn, phỏng vấn |
| 3. DFS stack | `(nút, mx)` trong stack | `O(n)` | `O(h)`–`O(n)` | Nộp bài Python an toàn |
| 4. BFS queue | `(nút, mx)` trong deque | `O(n)` | `O(w)` | Khi cần duyệt theo tầng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output):** `goodNodes([2,None,4,10,8,None,None,4])` bằng bao nhiêu?

**Bài 2 (tính tay):** liệt kê **giá trị** các nút tốt của ví dụ 1 theo thứ tự preorder (gốc → trái → phải).

**Bài 3 (điền TODO):** hoàn thiện `good_todo` (DFS lặp bằng stack).

**Bài 4 (tự viết lại):** viết `count_records(root)` đếm số nút có giá trị **lớn hơn hẳn** mọi tổ tiên (gốc luôn được tính).

**Bài 5 (mini challenge):** viết `max_ancestor_diff(root)` trả về `max |a.val − b.val|` với `a` là tổ tiên của `b` (LeetCode 1026; cây có ít nhất 2 nút), độ phức tạp `O(n)`.

In [24]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def good_todo(root):
    count = 0
    stack = [(root, root.val)]
    while stack:
        node, mx = stack.pop()
        # TODO: nếu node tốt thì tăng count
        # TODO: tính mx mới cho các con
        # TODO: đẩy các con (khác None) cùng mx mới vào stack
        pass
    return count

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Vẽ cây: 2 → phải 4; 4 có con 10 và 8; 8 có con trái 4. (2) Theo dõi max dọc đường. (3) Nút 4 cuối có max đường đi là 8.

**Bài 2:** (1) Preorder: 3, 1, 3, 4, 1, 5. (2) Đánh dấu nút tốt theo bảng ở mục 4. (3) Kết quả 4 phần tử.

**Bài 3:** (1) `if node.val >= mx: count += 1`. (2) `mx2 = max(mx, node.val)`. (3) Duyệt `(node.left, node.right)`, bỏ `None`.

**Bài 4:** (1) Giống bài gốc nhưng điều kiện `>`. (2) Gốc không có tổ tiên ⇒ khởi tạo `mx = −∞`. (3) Kiểm chứng bằng brute lưu đường đi.

**Bài 5:** (1) Với nút `b`, hiệu lớn nhất với tổ tiên là `max(|b − max_path|, |b − min_path|)`. (2) Mang cả `(lo, hi)` của đường đi. (3) Hoặc: tại lá, đáp án ứng viên là `hi − lo` của cả đường đi.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
ans1 = Solution().goodNodes(build_tree([2, None, 4, 10, 8, None, None, 4]))
print("Bài 1:", ans1)
assert ans1 == good_brute(build_tree([2, None, 4, 10, 8, None, None, 4])) == 4

# Bài 2
def good_values_preorder(root):
    out = []
    def dfs(node, mx):
        if node is None:
            return
        if node.val >= mx:
            out.append(node.val)
        dfs(node.left, max(mx, node.val))
        dfs(node.right, max(mx, node.val))
    dfs(root, float('-inf'))
    return out

print("Bài 2:", good_values_preorder(build_tree([3, 1, 4, 3, None, 1, 5])))
assert good_values_preorder(build_tree([3, 1, 4, 3, None, 1, 5])) == [3, 3, 4, 5]

# Bài 3
def good_todo_sol(root):
    count = 0
    stack = [(root, root.val)]
    while stack:
        node, mx = stack.pop()
        if node.val >= mx:
            count += 1
        mx2 = max(mx, node.val)
        for c in (node.left, node.right):
            if c:
                stack.append((c, mx2))
    return count

for arr, want in examples:
    assert good_todo_sol(build_tree(arr)) == want
print("Bài 3: OK")

Bài 1: 4
Bài 2: [3, 3, 4, 5]
Bài 3: OK


In [26]:
# Bài 4
def count_records(root):
    count, stack = 0, [(root, float('-inf'))]
    while stack:
        node, mx = stack.pop()
        if node.val > mx:
            count += 1
        for c in (node.left, node.right):
            if c:
                stack.append((c, max(mx, node.val)))
    return count

def count_records_brute(root):
    count, stack = 0, [(root, [])]
    while stack:
        node, path = stack.pop()
        count += all(v < node.val for v in path)
        for c in (node.left, node.right):
            if c:
                stack.append((c, path + [node.val]))
    return count

# Bài 5
def max_ancestor_diff(root):
    best, stack = 0, [(root, root.val, root.val)]
    while stack:
        node, lo, hi = stack.pop()
        lo, hi = min(lo, node.val), max(hi, node.val)
        best = max(best, hi - lo)
        for c in (node.left, node.right):
            if c:
                stack.append((c, lo, hi))
    return best

def max_ancestor_diff_brute(root):
    best, stack = 0, [(root, [])]
    while stack:
        node, path = stack.pop()
        for v in path:
            best = max(best, abs(v - node.val))
        for c in (node.left, node.right):
            if c:
                stack.append((c, path + [node.val]))
    return best

assert max_ancestor_diff(build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])) == 7
rng = random.Random(19)
for _ in range(500):
    tree = random_tree(rng.randint(2, 30), rng, -20, 20)
    assert count_records(tree) == count_records_brute(tree)
    assert max_ancestor_diff(tree) == max_ancestor_diff_brute(tree)
print("Bài 4, 5: OK (500 cây ngẫu nhiên)")

Bài 4, 5: OK (500 cây ngẫu nhiên)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "mang thông tin đường đi xuống" bằng stack:**

```python
count, stack = 0, [(root, root.val)]
while stack:
    node, mx = stack.pop()
    if node.val >= mx:
        count += 1
        mx = node.val
    if node.left:  stack.append((node.left, mx))
    if node.right: stack.append((node.right, mx))
return count
```

**Template đệ quy:**

```python
def dfs(node, mx):
    if not node: return 0
    mx2 = max(mx, node.val)
    return (node.val >= mx) + dfs(node.left, mx2) + dfs(node.right, mx2)
```

**Checklist:**
- [ ] So với **max của cả đường đi**, không chỉ cha.
- [ ] Dùng `>=` (bằng nhau vẫn tốt).
- [ ] Khởi tạo `mx` bằng `root.val` hoặc `−∞`, không phải 0.
- [ ] `mx` riêng cho từng nhánh (tham số / phần tử stack), không dùng biến chung.
- [ ] Đã nghĩ tới cây lệch `10^5` tầng (dùng stack hoặc nâng recursion limit).